In [44]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import roc_auc_score, average_precision_score


In [45]:
PROJECT_ROOT = Path("..").resolve()   # notebook 在 notebooks/ 里
DATA_DIR = PROJECT_ROOT / "data"

df_feat = pd.read_parquet(DATA_DIR / "df_feat.parquet")
df_feat.shape


(7812, 141)

In [ ]:
import sys
sys.path.append(str(PROJECT_ROOT))

from src.models import split_train_val_test
from src.models import make_preprocess  

In [47]:
W_COL = "LONGWT"
STR_COL = "VARSTR"
PSU_COL = "VARPSU"

df_feat[[W_COL, STR_COL, PSU_COL]].isna().sum()


LONGWT    0
VARSTR    0
VARPSU    0
dtype: int64

In [6]:
cat_cols = [
    "RACE_ETH",
    "REGIONY1_CAT",
    "EDU_GROUP",
    "POVCATY1_CAT",
    "FAMSIZE_Y1_GRP",
    "INS_TYPE_Y1",
]

In [7]:
#Numeric (use engineered columns, not raw)

num_cols = [
    # demographics / SES
    "AGE",
    "SEX_BIN",
    "LOG_FAMINCY1",
    "FAMSIZE_Y1",

   

    # employment
    "WORKED_Y1",
    "ANY_UNEMP_COMP_Y1",
    "LOG_UNEMP_COMP_Y1",
    "EMP_INFO_R12",
    "EMP_ATTACHED_ANY_R12_FILL0",  # model-friendly version

    # health status baseline
    "RTHLTH1_FAIRPOOR",
    "MNHLTH1_FAIRPOOR",

    # chronic conditions baseline
    "HIBPDXY1_BIN",
    "CHDDXY1_BIN",
    "STRKDXY1_BIN",
    "CHOLDXY1_BIN",
    "ASTHDXY1_BIN",
    "DIABDXY1_M18_BIN",
    # "MULTIMORBIDITY_Y1",   # optional (can remove if you keep all *_BIN)

    # baseline utilisation/cost
    "LOG_TOTEXPY1",
    "ANY_ED_Y1",
    "ANY_IP_Y1",
]

In [8]:
FEATURES = num_cols + cat_cols


In [29]:
def weighted_clf_metrics(y_true, proba, w, threshold):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba).astype(float)
    w = np.asarray(w).astype(float)

    auc_w = roc_auc_score(y_true, proba, sample_weight=w)
    pr_w = average_precision_score(y_true, proba, sample_weight=w)

    pred = (proba >= threshold).astype(int)
    tp = w[(y_true==1) & (pred==1)].sum()
    fp = w[(y_true==0) & (pred==1)].sum()
    fn = w[(y_true==1) & (pred==0)].sum()

    precision_w = tp / (tp + fp + 1e-12)
    recall_w = tp / (tp + fn + 1e-12)

    return {"AUC_w": float(auc_w), "PR_AUC_w": float(pr_w),
            "precision_w": float(precision_w), "recall_w": float(recall_w)}


In [30]:
def build_split_with_weights(df, target, feature_cols, *, random_state=42, stratify=False):
    tmp = df[feature_cols + [target, W_COL]].dropna()
    X = tmp[feature_cols].copy()
    y = tmp[target].copy()

    X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
        X, y, random_state=random_state, stratify=stratify
    )
    w_test = tmp.loc[X_test.index, W_COL].astype(float).values
    return X_train, X_val, X_test, y_train, y_val, y_test, w_test


In [40]:
RANDOM_SEED = 42

# preprocessing（树模型不需要scale）
pre_tree = make_preprocess(num_cols,cat_cols, scale_numeric=False)

# ---- HIGHCOST RF ----
X_tr, X_va, X_te, y_tr, y_va, y_te, w_te = build_split_with_weights(
    df_feat, "HIGHCOST_Y2", FEATURES, random_state=RANDOM_SEED, stratify=True
)
rf_hc = RandomForestClassifier(
    n_estimators=800, max_depth=16, min_samples_leaf=3,
    random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"
)
pipe_hc = Pipeline([("preprocess", pre_tree), ("model", rf_hc)])
pipe_hc.fit(X_tr, y_tr)
proba_hc = pipe_hc.predict_proba(X_te)[:, 1]

t_hc = 0.55
hc_w = weighted_clf_metrics(y_te.values, proba_hc, w_te, threshold=t_hc)
hc_w


{'AUC_w': 0.849763335535662,
 'PR_AUC_w': 0.39147541902771293,
 'precision_w': 0.4964422847105858,
 'recall_w': 0.38825690685445563}

In [41]:
# ---- ANY_ED XGB ----
X_tr, X_va, X_te, y_tr, y_va, y_te, w_te = build_split_with_weights(
    df_feat, "ANY_ED_Y2", FEATURES, random_state=RANDOM_SEED, stratify=True
)

xgb_ed = XGBClassifier(
    n_estimators=800, max_depth=3, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    random_state=RANDOM_SEED, n_jobs=-1, tree_method="hist",
    eval_metric="logloss"
)
pipe_ed = Pipeline([("preprocess", pre_tree), ("model", xgb_ed)])
pipe_ed.fit(X_tr, y_tr)
proba_ed = pipe_ed.predict_proba(X_te)[:, 1]

t_ed = 0.20
ed_w = weighted_clf_metrics(y_te.values, proba_ed, w_te, threshold=t_ed)
ed_w


{'AUC_w': 0.7000538014536916,
 'PR_AUC_w': 0.2868167063671123,
 'precision_w': 0.2677941143346467,
 'recall_w': 0.4588862270083041}

In [42]:
# ---- ANY_IP RF ----
X_tr, X_va, X_te, y_tr, y_va, y_te, w_te = build_split_with_weights(
    df_feat, "ANY_IP_Y2", FEATURES, random_state=RANDOM_SEED, stratify=True
)

rf_ip = RandomForestClassifier(
    n_estimators=800, max_depth=16, min_samples_leaf=3,
    random_state=RANDOM_SEED, n_jobs=-1, class_weight="balanced_subsample"
)
pipe_ip = Pipeline([("preprocess", pre_tree), ("model", rf_ip)])
pipe_ip.fit(X_tr, y_tr)
proba_ip = pipe_ip.predict_proba(X_te)[:, 1]

t_ip = 0.40
ip_w = weighted_clf_metrics(y_te.values, proba_ip, w_te, threshold=t_ip)
ip_w


{'AUC_w': 0.7476741628670012,
 'PR_AUC_w': 0.19218476856949024,
 'precision_w': 0.2400953070219936,
 'recall_w': 0.23567980744083095}

Weighted evaluation using LONGWT produced slightly lower AUC/PR-AUC compared with the unweighted test evaluation, while precision–recall trade-offs shifted depending on the fixed operating threshold. Overall conclusions were unchanged: high-cost prediction remains the strongest task, ED is moderate, and inpatient admission is the most challenging due to rarity.